##### Import the libraries


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

##### Load the dataset

In [2]:
bd = pd.read_csv(r"C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Hospital_Bed_Demand_Forecasting\Data\Cleaned\bed_inventory_cleaned.csv")

In [3]:
bd.head()

,datetime,hospital_id,ward,bed_type,total_beds,staffed_beds,occupied_beds,closed_beds,occupancy_rate,bedding_year,bedding_month,bedding_quarter,bedding_weekday,bedding_hour
0,2024-01-01,HHN-BIR-01,Cardiology Ward,Standard,42,42,0,0,0.0,2024,January,1,Monday,0
1,2024-01-01,HHN-BIR-01,Cardiology Ward,Standard,42,42,0,0,0.0,2024,January,1,Monday,1
2,2024-01-01,HHN-BIR-01,Cardiology Ward,Standard,42,42,0,0,0.0,2024,January,1,Monday,2
3,2024-01-01,HHN-BIR-01,Cardiology Ward,Standard,42,42,0,0,0.0,2024,January,1,Monday,3
4,2024-01-01,HHN-BIR-01,Cardiology Ward,Standard,42,42,0,0,0.0,2024,January,1,Monday,4


##### Convert hourly records into daily records.

In [4]:
bd["datetime"] = pd.to_datetime(bd["datetime"])
bd["date"] = bd["datetime"].dt.floor("D")

In [5]:
bd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 701760 entries, 0 to 701759
Data columns (total 15 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   datetime         701760 non-null  datetime64[ns]
 1   hospital_id      701760 non-null  object        
 2   ward             701760 non-null  object        
 3   bed_type         701760 non-null  object        
 4   total_beds       701760 non-null  int64         
 5   staffed_beds     701760 non-null  int64         
 6   occupied_beds    701760 non-null  int64         
 7   closed_beds      701760 non-null  int64         
 8   occupancy_rate   701760 non-null  float64       
 9   bedding_year     701760 non-null  int64         
 10  bedding_month    701760 non-null  object        
 11  bedding_quarter  701760 non-null  int64         
 12  bedding_weekday  701760 non-null  object        
 13  bedding_hour     701760 non-null  int64         
 14  date             701

In [6]:
bd["ward"].unique()

array(['Cardiology Ward', 'Day Case Unit', 'General Medicine Ward A',
       'General Medicine Ward B', 'ICU', 'Oncology Ward',
       'Orthopaedics Ward A', 'Orthopaedics Ward B'], dtype=object)

In [7]:
daily_units = (bd.groupby(["hospital_id","datetime","ward"],as_index=False)
            .agg({"occupied_beds":"mean", "staffed_beds":"mean", "occupancy_rate":"mean" }))

In [8]:
ward_series = {}

for ward in daily_units["ward"].unique():

    ward_series[ward] = (daily_units[daily_units["ward"] == ward].sort_values(["hospital_id", "datetime"])
        .set_index("datetime"))

In [9]:
daily_units.head(10)

,hospital_id,datetime,ward,occupied_beds,staffed_beds,occupancy_rate
0,HHN-BIR-01,2024-01-01,Cardiology Ward,3.083333,41.833333,0.073895
1,HHN-BIR-01,2024-01-01,Day Case Unit,0.000000,2.916667,0.000000
2,HHN-BIR-01,2024-01-01,General Medicine Ward A,0.458333,15.833333,0.029018
3,HHN-BIR-01,2024-01-01,General Medicine Ward B,1.583333,15.916667,0.099306
4,HHN-BIR-01,2024-01-01,ICU,2.041667,8.958333,0.227431
5,HHN-BIR-01,2024-01-01,Oncology Ward,0.000000,17.916667,0.000000
6,HHN-BIR-01,2024-01-01,Orthopaedics Ward A,1.375000,10.875000,0.126136
7,HHN-BIR-01,2024-01-01,Orthopaedics Ward B,0.125000,10.791667,0.011364
8,HHN-BIR-01,2024-01-02,Cardiology Ward,10.333333,41.458333,0.248624
9,HHN-BIR-01,2024-01-02,Day Case Unit,0.625000,2.875000,0.215278


In [10]:
daily_units["ward"].unique()

array(['Cardiology Ward', 'Day Case Unit', 'General Medicine Ward A',
       'General Medicine Ward B', 'ICU', 'Oncology Ward',
       'Orthopaedics Ward A', 'Orthopaedics Ward B'], dtype=object)

In [11]:
icu = daily_units[daily_units["ward"] == "ICU"]

icu.head(10)

,hospital_id,datetime,ward,occupied_beds,staffed_beds,occupancy_rate
4,HHN-BIR-01,2024-01-01,ICU,2.041667,8.958333,0.227431
12,HHN-BIR-01,2024-01-02,ICU,4.583333,8.833333,0.519676
20,HHN-BIR-01,2024-01-03,ICU,4.916667,9.000000,0.546296
28,HHN-BIR-01,2024-01-04,ICU,5.000000,8.958333,0.558449
36,HHN-BIR-01,2024-01-05,ICU,5.166667,8.958333,0.577546
44,HHN-BIR-01,2024-01-06,ICU,6.291667,8.916667,0.706019
52,HHN-BIR-01,2024-01-07,ICU,8.958333,8.958333,1.000000
60,HHN-BIR-01,2024-01-08,ICU,8.916667,8.916667,1.000000
68,HHN-BIR-01,2024-01-09,ICU,8.916667,8.916667,1.000000
76,HHN-BIR-01,2024-01-10,ICU,8.791667,8.791667,1.000000


##### Create separate time series for each unit

In [12]:
icu = daily_units[daily_units["ward"] == "ICU"]
daycase = daily_units[daily_units["ward"] == "Day Case Unit"]
cardiology = daily_units[daily_units["ward"] == "Cardiology Ward"]
general_A = daily_units[daily_units["ward"] == "General Medicine Ward A"]
general_B = daily_units[daily_units["ward"] == "General Medicine Ward B"]
oncology = daily_units[daily_units["ward"] == "Oncology Ward"]
ortho_A = daily_units[daily_units["ward"] == "Orthopaedics Ward A"]
ortho_B = daily_units[daily_units["ward"] == "Orthopaedics Ward B"]

In [13]:
icu.to_csv("../Data/TimeSeries/ICU_TimeSeries.csv", index=False)

general_A.to_csv("../Data/TimeSeries/General_Medicine_Ward_A_TimeSeries.csv", index=False)

general_B.to_csv("../Data/TimeSeries/General_Medicine_Ward_B_TimeSeries.csv", index=False)

cardiology.to_csv("../Data/TimeSeries/Cardiology_Ward_TimeSeries.csv", index=False)

oncology.to_csv("../Data/TimeSeries/Oncology_Ward_TimeSeries.csv", index=False)

ortho_A.to_csv("../Data/TimeSeries/Orthopaedics_Ward_A_TimeSeries.csv", index=False)

ortho_B.to_csv("../Data/TimeSeries/Orthopaedics_Ward_B_TimeSeries.csv", index=False)

daycase.to_csv("../Data/TimeSeries/Day_Case_Unit_TimeSeries.csv", index=False)